# Imports

In [1]:
import os
import pertpy as pt
import scanpy as sc
import numpy as np
import plotly.express as px
import pandas as pd
import scipy.sparse as sp
import einops
import umap
import gseapy as gp
from gseapy import enrichr
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scbmlp.datasets import get_regression_datasets
from scbmlp.models import ScBMLPRegressor, Config

In [2]:
# Set random seeds for reproducibility
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)  # For CUDA if available
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set scanpy settings for deterministic behavior
sc.settings.verbosity = 2  # Reduce verbosity
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [3]:
# Create folder to store figures
os.makedirs("figures/perturbation/", exist_ok=True)

# Set figure params for blog dimensions
fig_width = 800
title_fontsize = 18
legend_fontsize = 14

# Load data

## Load

In [4]:
# n_genes = 10_000
n_genes = 5_000  # reduce to mitigate overfitting
device = "cpu"
random_state = 42
target_key = "perturbation"

adata = pt.data.srivatsan_2020_sciplex2()

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, subset=True, n_top_genes=n_genes)

normalizing counts per cell
    finished (0:00:00)
extracting highly variable genes
    finished (0:00:00)


In [5]:
adata

AnnData object with n_obs × n_vars = 24262 × 5000
    obs: 'ncounts', 'hash_umis', 'pval_demultiplexing', 'qval_demultiplexing', 'top_to_second_best_ratio', 'top_oligo', 'perturbation', 'dose_value', 'well', 'celltype', 'cell_line', 'cancer', 'disease', 'tissue_type', 'organism', 'perturbation_type', 'ngenes', 'percent_mito', 'percent_ribo', 'nperts', 'chembl-ID'
    var: 'ensembl_id', 'ncounts', 'ncells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

## Clean

In [6]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

computing PCA
    with n_comps=50
    finished (0:00:01)
computing neighbors
    using 'X_pca' with n_pcs = 50
    finished (0:00:11)
computing UMAP
    finished (0:00:11)


In [7]:
# Make cell type colormap
unique_clusters = adata.obs["perturbation"].unique()

# colors = px.colors.qualitative.Set1[:len(unique_clusters)]  # High contrast, scientific
# colors = px.colors.qualitative.Dark2[:len(unique_clusters)]  # Darker colors
# colors = px.colors.qualitative.T10[:len(unique_clusters)]    # Tableau colors
# colors = px.colors.qualitative.Pastel1[:len(unique_clusters)]  # Softer colors
colors = px.colors.qualitative.Plotly[:len(unique_clusters)]  # Original Plotly colors
# colors = sc.pl.palettes.default_28[:len(unique_clusters)]     # Scanpy default

# import plotly.colors as pc
# colors = pc.sample_colorscale(cmap, [i/(len(unique_clusters)-1) for i in range(len(unique_clusters))])

cluster_colors = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}

In [8]:
# Remove control since it's represented as a zero vector
adata = adata[adata.obs["perturbation"] != "control"].copy()

In [9]:
fig = px.scatter(
    x=adata.obsm["X_umap"][:, 0],
    y=adata.obsm["X_umap"][:, 1],
    color=adata.obs[target_key],
    title="Perturbations in UMAP Space",
    labels={"x": "UMAP1", "y": "UMAP2"},
    width=fig_width,
    height=700,
    color_discrete_map=cluster_colors,
    category_orders={"perturbation": list(cluster_colors.keys())},
)
fig.update_traces(marker=dict(size=3))

# Apply custom font sizes from earlier variables
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=150, r=150, t=120, b=120)
)
# Axis title and tick font tuning (slightly smaller than legend for balance)
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save plot (keep higher resolution export)
# fig.write_html("figures/perturbation/perturbation_umap.html")
# fig.write_image("figures/perturbation/perturbation_umap.png", width=fig_width, scale=2)

In [10]:
fig = px.scatter(
    x=adata.obsm["X_umap"][:, 0],
    y=adata.obsm["X_umap"][:, 1],
    color=adata.obs["ncounts"],
    color_continuous_scale="icefire",
    title="Perturbations in UMAP Space",
    labels={"x": "UMAP1", "y": "UMAP2"},
    width=fig_width,
    height=700,
)
fig.update_traces(marker=dict(size=3))

# Apply custom font sizes from earlier variables
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=150, r=150, t=120, b=120)
)
# Axis title and tick font tuning (slightly smaller than legend for balance)
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save plot (keep higher resolution export)
# fig.write_html("figures/perturbation/perturbation_umap.html")
# fig.write_image("figures/perturbation/dose_umap.png", width=fig_width, scale=2)

Cant separate cells with low counts.

In [11]:
px.histogram(adata.obs["ncounts"], nbins=50, title="n_counts Distribution", width=600, height=300)

In [12]:
adata

AnnData object with n_obs × n_vars = 23733 × 5000
    obs: 'ncounts', 'hash_umis', 'pval_demultiplexing', 'qval_demultiplexing', 'top_to_second_best_ratio', 'top_oligo', 'perturbation', 'dose_value', 'well', 'celltype', 'cell_line', 'cancer', 'disease', 'tissue_type', 'organism', 'perturbation_type', 'ngenes', 'percent_mito', 'percent_ribo', 'nperts', 'chembl-ID'
    var: 'ensembl_id', 'ncounts', 'ncells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [13]:
counts_low_thresh = 5_000
counts_high_thresh = 20_000
adata = adata[adata.obs["ncounts"].between(counts_low_thresh, counts_high_thresh)]

In [14]:
adata

View of AnnData object with n_obs × n_vars = 9689 × 5000
    obs: 'ncounts', 'hash_umis', 'pval_demultiplexing', 'qval_demultiplexing', 'top_to_second_best_ratio', 'top_oligo', 'perturbation', 'dose_value', 'well', 'celltype', 'cell_line', 'cancer', 'disease', 'tissue_type', 'organism', 'perturbation_type', 'ngenes', 'percent_mito', 'percent_ribo', 'nperts', 'chembl-ID'
    var: 'ensembl_id', 'ncounts', 'ncells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [15]:
fig = px.scatter(
    x=adata.obsm["X_umap"][:, 0],
    y=adata.obsm["X_umap"][:, 1],
    color=adata.obs["ncounts"],
    color_continuous_scale="icefire",
    title="Perturbations in UMAP Space",
    labels={"x": "UMAP1", "y": "UMAP2"},
    width=fig_width,
    height=700,
)
fig.update_traces(marker=dict(size=3))

# Apply custom font sizes from earlier variables
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=150, r=150, t=120, b=120)
)
# Axis title and tick font tuning (slightly smaller than legend for balance)
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save plot (keep higher resolution export)
# fig.write_html("figures/perturbation/perturbation_umap.html")
# fig.write_image("figures/perturbation/dose_umap.png", width=fig_width, scale=2)

In [16]:
fig = px.scatter(
    x=adata.obsm["X_umap"][:, 0],
    y=adata.obsm["X_umap"][:, 1],
    color=adata.obs["perturbation"],
    title="Perturbation Type in UMAP Space",
    labels={"x": "UMAP1", "y": "UMAP2"},
    width=fig_width,
    height=400,
    color_discrete_map=cluster_colors,
    category_orders={"perturbation": list(cluster_colors.keys())},
)
fig.update_traces(marker=dict(size=3))

# Apply custom font sizes from earlier variables
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=250, r=250, t=100, b=50)
)

# Axis title and tick font tuning (slightly smaller than legend for balance)
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save plot (keep higher resolution export)
# fig.write_html("figures/perturbation/perturbation_umap.html")
fig.write_image("figures/perturbation/perturbation_umap.png", width=fig_width, height=400, scale=2)

In [17]:
doses = adata.obs["dose_value"].values.astype(float)
log_doses = np.log10(1+doses)
fig = px.scatter(
    x=adata.obsm["X_umap"][:, 0],
    y=adata.obsm["X_umap"][:, 1],
    color=log_doses,
    color_continuous_scale="Reds",
    title="Perturbation Strength (log1p base 10) in UMAP Space",
    labels={"x": "UMAP1", "y": "UMAP2"},
    width=fig_width,
    height=400,
)
fig.update_traces(marker=dict(size=3))

# Apply custom font sizes from earlier variables
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=250, r=250, t=100, b=50)
)

# Axis title and tick font tuning (slightly smaller than legend for balance)
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

# Update colorbar title
fig.update_coloraxes(colorbar_title_text="Perturbation Strength")
fig.update_layout(coloraxis_colorbar_title_side="right")
 
fig.show()

# Save plot (keep higher resolution export)
# fig.write_html("figures/perturbation/perturbation_umap.html")
fig.write_image("figures/perturbation/dose_umap.png", width=fig_width, height=400, scale=2)

As one might expect, the stronger perturbations appear easier to predict.

## Create dataset

In [18]:
cats = adata.obs[target_key].astype('category')
one_hot = np.eye(len(cats.cat.categories))[cats.cat.codes.to_numpy()]
dose = adata.obs['dose_value'].astype(float).to_numpy()
adata.obsm['perturbation'] = one_hot * np.log10(1.0 + dose)[:, None]
adata.obsm['perturbation'] = np.nan_to_num(adata.obsm['perturbation'], nan=0.0)

/var/folders/pg/n6mr0f253l97bcvf2b4gjtzw0000gn/T/ipykernel_51895/3175205223.py:4: ImplicitModificationWarning:

Setting element `.obsm['perturbation']` of view, initializing view as actual.



In [19]:
adata.obsm["perturbation"]

array([[0.        , 1.70757018, 0.        , 0.        ],
       [0.17609126, 0.        , 0.        , 0.        ],
       [0.04139269, 0.        , 0.        , 0.        ],
       ...,
       [0.        , 0.        , 0.04139269, 0.        ],
       [0.        , 0.77815125, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 1.04139269]], shape=(9689, 4))

In [20]:
adata.obs[["perturbation", "dose_value"]]

,perturbation,dose_value
cell_barcode,,
A01_A01_RT_396,Dex,50
A01_A01_RT_397,BMS,0.5
A01_A01_RT_401,BMS,0.1
A01_A01_RT_402,Nutlin,10
A01_A01_RT_403,SAHA,0.5
...,...,...
H12_B02_RT_734,BMS,0
H12_B02_RT_752,BMS,0.1
H12_B02_RT_753,Nutlin,0.1


In [21]:
train_dataset, val_dataset, _ = get_regression_datasets(
    adata,
    target_key=target_key,
    train_split=0.7,
    val_split=0.3,
    random_state=random_state,
    device=device,
)

# Train

In [22]:
d_hidden = 128
n_epochs = 100
lr = 1e-4
device = "cpu"
batch_size = 128
n_perturbations = adata.obs["perturbation"].nunique()

In [23]:
cfg = Config(
    d_input=n_genes,
    d_hidden=d_hidden,
    d_output=n_perturbations,
    n_epochs=n_epochs,
    lr=lr,
    device=device,
    batch_size=batch_size,
    bias=True,
    seed=random_state,
    dropout_rate=0.4,
    weight_decay=1e-3,
)

model = ScBMLPRegressor(cfg, loss_fn="l1")
train_losses, train_metrics, val_losses, val_metrics = model.fit(
    train_dataset, val_dataset,
)

Training for 100 epochs: 100%|██████████| 100/100 [00:40<00:00,  2.47it/s, train_loss=0.0811, train_mae=0.0811, val_loss=0.0971, val_mae=0.0971]


In [24]:
# Create combined plot with loss and MAE subplots

# Prepare data
loss_df = pd.DataFrame({
    'Epoch': list(range(len(train_losses))) + list(range(len(val_losses))),
    'Loss': train_losses + val_losses,
    'Type': ['Train'] * len(train_losses) + ['Validation'] * len(val_losses)
})

metric_df = pd.DataFrame({
    'Epoch': list(range(len(train_metrics))) + list(range(len(val_metrics))),
    'MAE': train_metrics + val_metrics,
    'Type': ['Train'] * len(train_metrics) + ['Validation'] * len(val_metrics)
})

# Create subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Training and Validation Loss', 'Training and Validation MAE'],
    vertical_spacing=0.15
)

# Colors shared across metrics
colors = {'Train': 'blue', 'Validation': 'red'}

# Add loss plot traces (these will own the legend entries)
for type_name in ['Train', 'Validation']:
    type_data = loss_df[loss_df['Type'] == type_name]
    fig.add_trace(
        go.Scatter(
            x=type_data['Epoch'],
            y=type_data['Loss'],
            mode='lines',
            name=type_name,               # Single legend entry per type
            legendgroup=type_name,
            showlegend=True,              # Only show legend for loss traces
            line=dict(color=colors[type_name], width=2),
            hovertemplate=f"Epoch %{{x}}<br>{type_name} Loss: %{{y:.4f}}<extra></extra>"
        ),
        row=1, col=1
    )

# Add MAE plot traces (no new legend entries, grouped with loss) - now solid lines
for type_name in ['Train', 'Validation']:
    type_data = metric_df[metric_df['Type'] == type_name]
    fig.add_trace(
        go.Scatter(
            x=type_data['Epoch'],
            y=type_data['MAE'],
            mode='lines',
            name=type_name,               # Same name but suppressed in legend
            legendgroup=type_name,
            showlegend=False,
            line=dict(color=colors[type_name], width=2),
            hovertemplate=f"Epoch %{{x}}<br>{type_name} MAE: %{{y:.4f}}<extra></extra>"
        ),
        row=2, col=1
    )

# Update layout (dimensions & base styling)
fig.update_layout(
    height=500,
    width=fig_width,
    title_text="Training Progress: Loss and MAE",
    showlegend=True,
    legend=dict(title=None, orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=80, r=80, t=80, b=60)
)

# Axis labels
fig.update_xaxes(title_text="Epoch", row=2, col=1)
fig.update_yaxes(title_text="Loss", row=1, col=1)
fig.update_yaxes(title_text="MAE", row=2, col=1)

# Apply custom font sizes
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize))
)

# Axis title and tick font tuning
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save plot
# fig.write_html("figures/perturbation/perturbation_training.html")
fig.write_image("figures/perturbation/perturbation_training.png", width=fig_width, height=600, scale=2)

In [25]:
# Get predictions on validation set to analyze error distribution
model.eval()
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in val_loader:
        inputs, targets = batch
        predictions = model(inputs)
        all_predictions.append(predictions.cpu())
        all_targets.append(targets.cpu())

# Concatenate all batches
predictions = torch.cat(all_predictions, dim=0)
targets = torch.cat(all_targets, dim=0)

# Calculate absolute errors per frequency component
absolute_errors = torch.abs(predictions - targets)  # Shape: (n_samples, n_freqs)

In [26]:
# Per-perturbation error analysis (refactored: 3 horizontal subplots)
# Subplots: (1) Per-perturbation MAE bar (top N if large) (2) Per-cell MAE histogram (3) Absolute error boxplots (top N if large)

# Absolute errors already computed: absolute_errors (n_samples, n_perturbations)
per_perturbation_mae = absolute_errors.mean(dim=0).numpy()
per_cell_mae = absolute_errors.mean(dim=1).numpy()

n_pert = per_perturbation_mae.shape[0]

# Derive perturbation names from categorical ordering
perturbation_categories = adata.obs[target_key].astype('category').cat.categories.tolist()
if len(perturbation_categories) != n_pert:
    perturbation_names = [f"Pert {i}" for i in range(n_pert)]
else:
    perturbation_names = [str(c) for c in perturbation_categories]

# Prevalence counts (not plotted now but can be useful later)
pert_counts = adata.obs[target_key].value_counts().reindex(perturbation_categories).fillna(0).to_numpy() if len(perturbation_categories)==n_pert else np.bincount(torch.argmax(targets, dim=1).numpy(), minlength=n_pert)

from plotly.subplots import make_subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Per-Perturbation MAE',
        'Per-Cell MAE Distribution',
        'Absolute Errors by Perturbation'
    ],
    specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "xy"}]],
    horizontal_spacing=0.10
)

# 1. Bar plot (cap to max_bar highest MAE for readability)
max_bar = 40
if n_pert > max_bar:
    top_idx = np.argsort(per_perturbation_mae)[-max_bar:][::-1]
    bar_x = [perturbation_names[i] for i in top_idx]
    bar_y = per_perturbation_mae[top_idx]
else:
    bar_x = perturbation_names
    bar_y = per_perturbation_mae
fig.add_trace(
    go.Bar(x=bar_x, y=bar_y, name="Per-Perturbation MAE"),
    row=1, col=1
)

# 2. Histogram of per-cell MAE
fig.add_trace(
    go.Histogram(x=per_cell_mae, nbinsx=50, name="Per-Cell MAE"),
    row=1, col=2
)

# 3. Box plots (cap to max_box highest MAE)
max_box = 25
if n_pert > max_box:
    top_idx_box = np.argsort(per_perturbation_mae)[-max_box:][::-1]
else:
    top_idx_box = np.arange(n_pert)
for idx in top_idx_box:
    fig.add_trace(
        go.Box(y=absolute_errors[:, idx].numpy(), name=perturbation_names[idx], showlegend=False),
        row=1, col=3
    )

# Layout & styling
fig.update_layout(
    height=450,
    width=fig_width,
    title_text="Perturbation Prediction Error Analysis",
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    showlegend=False,
    margin=dict(l=30, r=30, t=70, b=50)
)

# Axis font sizing
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))

# Axes labels
fig.update_xaxes(title_text="Perturbation", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=1)
fig.update_yaxes(title_text="MAE", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=1)
fig.update_xaxes(title_text="MAE", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=2)
fig.update_yaxes(title_text="Count", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=2)
fig.update_xaxes(title_text="Perturbation", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=3)
fig.update_yaxes(title_text="Absolute Error", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=3)

fig.show()

# Save plot
# fig.write_html("figures/perturbation/perturbation_model_analysis.html")
fig.write_image("figures/perturbation/perturbation_model_analysis.png", width=fig_width, height=300, scale=2)

In [ ]:
# Plot bias distributions to check if they are being used
px.histogram(model.left.bias).show()
px.histogram(model.right.bias).show()

# Weight interpretation

## Setup

In [27]:
def get_marker_gene_lists(
    gene_names: np.ndarray,
    vecs: np.ndarray,
    n_modules: int = 1,
    n_top_genes: int = 50,
) -> np.ndarray:
    """Extract marker genes optimized for GO analysis."""
    gene_lists = []
    for i in range(n_modules):
        top_idxs = vecs[:,i].topk(n_top_genes).indices
        top_genes = gene_names[top_idxs].tolist()
        bottom_idxs = (-vecs[:,i]).topk(n_top_genes).indices
        bottom_genes = gene_names[bottom_idxs].tolist()
        gene_lists.append([top_genes, bottom_genes])
    return np.array(gene_lists)

def decompose_gene_weights(model, out_idx):
    """Perform eigendecomposition of bilinear weights for a specific output."""
    q = einops.einsum(model.w_p[out_idx], model.w_l, model.w_r, "hid, hid in1, hid in2 -> in1 in2")
    q = 0.5 * (q + q.T)  # symmetrize
    vals, vecs = torch.linalg.eigh(q)
    vals = vals.flip([0])
    vecs = vecs.flip([1])
    return vals, vecs


def print_gene_modules(gene_name, gene_lists, n_modules=3):
    """Print gene modules for a specific gene."""
    print("="*20, gene_name, "="*20)
    for module_idx in range(n_modules):
        print("="*20, "Module", module_idx, "="*20)
        print(f"Positive genes: {gene_lists[module_idx,0,:10]}...")
        print(f"Negative genes: {gene_lists[module_idx,1,:10]}...")


def analyze_go_terms(gene_lists, n_modules=3, n_results=5):
    """Perform GO term analysis on gene modules."""
    print("\n" + "="*50)
    print("GO ANALYSIS")
    print("="*50)

    results_cols = ["Term", "Genes", "Gene_set", "Adjusted P-value"]

    for module_idx in range(n_modules):
        print("="*10, "Module", module_idx, "="*10)

        # Use Enrichr with optimized gene sets for pancreatic development
        for i in range(2):
            side = "Positive" if i == 0 else "Negative"
            print(f"\n--- {side} genes ---")
            enr = gp.enrichr(
                gene_list=gene_lists[module_idx, i].tolist(),
                gene_sets=[
                    "GO_Biological_Process_2023",
                    "GO_Molecular_Function_2023",
                    "GO_Cellular_Component_2023", 
                    "KEGG_2019_Human",
                    "Reactome_2022",
                    "MSigDB_Hallmark_2020",
                    "WikiPathways_2019_Human"
                ],
                organism="human",
            )
            # Filter and combine results from all gene sets
            all_results = enr.results[enr.results['Adjusted P-value'] <= 0.05].copy()
            if len(all_results) > 0:
                # Sort by p-value and show top results
                all_results = all_results.sort_values('Adjusted P-value')
                display(all_results.head(n_results)[results_cols])
            else:
                print(f"No significant results found (p < 0.05)")

In [28]:
# Shared parameters for frequency module analysis
n_modules = 3
n_top_genes = int(0.01*n_genes)  # top 1% of genes
gene_names = adata.var_names.values
n_results = 15

In [29]:
pert2idx = {p: i for i, p in enumerate(adata.obs['perturbation'].cat.categories)}
idx2pert = {i: p for p, i in pert2idx.items()}

## BMS

In [30]:
pert_idx = 0
pert = idx2pert[pert_idx]
print(pert)

BMS


In [31]:
# Decompose weights and extract gene modules
vals, vecs = decompose_gene_weights(model, pert_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"{idx2pert[pert_idx]}", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== BMS ====================
==================== Module 0 ====================
Positive genes: ['DKK1' 'TP53I3' 'FAM129A' 'SREBF1' 'SEMA3A' 'ASTN2' 'Z83843.1' 'HGD'
 'FKBP5' 'FGA']...
Negative genes: ['AC027288.3' 'PHLDA2' 'OPCML' 'ARHGAP15' 'SERTAD2' 'AC011287.1' 'FRMD4A'
 'CATSPERB' 'AC005614.2' 'SCUBE3']...
==================== Module 1 ====================
Positive genes: ['SCUBE3' 'SH3BP2' 'AC011287.1' 'HSPA1A' 'TUBA1B' 'MKI67' 'AC016205.1'
 'PDE4B' 'MACROD2' 'PDE7B']...
Negative genes: ['CDKN1A' 'CLU' 'HIF1A-AS2' 'DKK1' 'MDM2' 'TRAM1' 'MT-TL2' 'SULF2'
 'IGFL2-AS1' 'SYBU']...
==================== Module 2 ====================
Positive genes: ['MDM2' 'SCUBE3' 'SSH2' 'FRMD5' 'CDKN1A' 'SH3BP2' 'TUBA1B' 'CERS6'
 'HIF1A-AS2' 'PTPRT']...
Negative genes: ['FGG' 'CLU' 'LINGO2' 'DKK1' 'KAZN' 'IGSF11' 'AC011287.1' 'FHIT' 'ERRFI1'
 'AC025627.1']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
945,Blood Clotting Cascade WP272,FGB;FGA;F5,WikiPathways_2019_Human,0.001734
728,Common Pathway Of Fibrin Clot Formation R-HSA-...,FGB;FGA;F5,Reactome_2022,0.004313
946,Nuclear Receptors Meta-Pathway WP2882,SREBF1;PDK4;SERPINB9;PTGS2;TNS4;FKBP5,WikiPathways_2019_Human,0.004770
947,VEGFA-VEGFR2 Signaling Pathway WP3888,FGB;FGA;NR4A1;PTGS2;DKK1,WikiPathways_2019_Human,0.006932
729,Formation Of Fibrin Clot (Clotting Cascade) R-...,FGB;FGA;F5,Reactome_2022,0.010598
597,Platelet Alpha Granule Lumen (GO:0031093),FGB;FGA;F5,GO_Cellular_Component_2023,0.012645
596,Extracellular Vesicle (GO:1903561),FGB;FGA;F5,GO_Cellular_Component_2023,0.012645
595,Extracellular Membrane-Bounded Organelle (GO:0...,FGB;FGA;F5,GO_Cellular_Component_2023,0.012645
0,Induction Of Bacterial Agglutination (GO:0043152),FGB;FGA,GO_Biological_Process_2023,0.015483
1,Regulation Of Type B Pancreatic Cell Prolifera...,NR4A1;NUPR1,GO_Biological_Process_2023,0.015483



--- Negative genes ---
No significant results found (p < 0.05)
========== Module 1 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1315,G2-M Checkpoint,TOP2A;SLC7A5;CENPE;POLQ;CENPF;TGFB1;MKI67,MSigDB_Hallmark_2020,0.000019
1346,Nuclear Receptors Meta-Pathway WP2882,SLC7A5;ALDH3A1;TGFB1;PDE4B;CYP1B1;SQSTM1;PPARG...,WikiPathways_2019_Human,0.000129
1347,Aryl Hydrocarbon Receptor Pathway WP2873,SLC7A5;ALDH3A1;TGFB1;CYP1B1,WikiPathways_2019_Human,0.000292
0,Kinetochore Assembly (GO:0051382),CENPE;CENPF;KNTC1,GO_Biological_Process_2023,0.000587
1,Sister Chromatid Segregation (GO:0000819),TOP2A;KIF18A;KIF18B;KNTC1,GO_Biological_Process_2023,0.000587
904,Kinetochore Microtubule (GO:0005828),CENPE;KIF18A;KNTC1,GO_Cellular_Component_2023,0.000806
905,Spindle Microtubule (GO:0005876),CENPE;KIF18A;KIF18B;KNTC1,GO_Cellular_Component_2023,0.000863
906,Astral Microtubule (GO:0000235),KIF18A;KIF18B,GO_Cellular_Component_2023,0.002100
1316,E2F Targets,TOP2A;CENPE;KIF18B;DIAPH3;MKI67,MSigDB_Hallmark_2020,0.002167
1070,RHO GTPases Activate Formins R-HSA-5663220,CENPE;CENPF;KIF18A;DIAPH3;KNTC1,Reactome_2022,0.002872



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
1054,Apoptosis,CDKN1A;DPYD;CLU;RHOB;BIRC3,MSigDB_Hallmark_2020,0.001206
0,Positive Regulation Of Protein Kinase Activity...,EPHA4;CDKN1A;EMP2;DKK1;CLU,GO_Biological_Process_2023,0.018212
1083,Androgen receptor signaling pathway WP138,CDKN1A;MDM2;RHOB,WikiPathways_2019_Human,0.021917
1082,Apoptosis WP254,MDM2;TNFRSF10B;BIRC3,WikiPathways_2019_Human,0.021917
1081,TP53 Network WP1742,CDKN1A;MDM2,WikiPathways_2019_Human,0.021917
1080,Primary Focal Segmental Glomerulosclerosis FSG...,CDKN1A;KRT8;DKK1,WikiPathways_2019_Human,0.021917
1079,miRNA Regulation of DNA Damage Response WP1530,CDKN1A;MDM2;TNFRSF10B,WikiPathways_2019_Human,0.021917
1078,DNA Damage Response WP707,CDKN1A;MDM2;TNFRSF10B,WikiPathways_2019_Human,0.021917
1084,miRNA regulation of p53 pathway in prostate ca...,MDM2;TNFRSF10B,WikiPathways_2019_Human,0.021917
599,Ephrin Receptor Activity (GO:0005003),EPHA4;EPHA7,GO_Molecular_Function_2023,0.026771


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
801,DNA Damage Response WP707,CDKN1A;RFC1;MDM2;TNFRSF10B,WikiPathways_2019_Human,0.001083
802,miRNA Regulation of DNA Damage Response WP1530,CDKN1A;RFC1;MDM2;TNFRSF10B,WikiPathways_2019_Human,0.001083
803,TP53 Network WP1742,CDKN1A;MDM2,WikiPathways_2019_Human,0.023829
804,Androgen receptor signaling pathway WP138,CDKN1A;MDM2;RHOB,WikiPathways_2019_Human,0.023829
805,miRNA regulation of p53 pathway in prostate ca...,MDM2;TNFRSF10B,WikiPathways_2019_Human,0.023829
778,Hypoxia,CA12;CDKN1A;GBE1;TGFBI,MSigDB_Hallmark_2020,0.035877
806,miRNA regulation of prostate cancer signaling ...,CDKN1A;MDM2,WikiPathways_2019_Human,0.037447
807,ATM Signaling Pathway WP2516,CDKN1A;MDM2,WikiPathways_2019_Human,0.041030
808,Bladder Cancer WP2828,CDKN1A;MDM2,WikiPathways_2019_Human,0.041030
809,Integrated Cancer Pathway WP1971,CDKN1A;MDM2,WikiPathways_2019_Human,0.043952



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
573,Adrenergic Receptor Binding (GO:0031690),SLC9A3R1;NEDD4;MAGI2,GO_Molecular_Function_2023,0.000596
664,Platelet Alpha Granule Lumen (GO:0031093),FGB;FGG;VEGFC;CLU,GO_Cellular_Component_2023,0.001489
665,Platelet Alpha Granule (GO:0031091),FGB;FGG;VEGFC;CLU,GO_Cellular_Component_2023,0.002417
991,Estrogen Response Early,SLC9A3R1;FASN;KAZN;FKBP5;TGM2,MSigDB_Hallmark_2020,0.003076
992,Cholesterol Homeostasis,ERRFI1;FASN;CLU,MSigDB_Hallmark_2020,0.009252
993,Epithelial Mesenchymal Transition,VEGFC;DKK1;SNTB1;TGM2,MSigDB_Hallmark_2020,0.011439
0,Purine-Containing Compound Metabolic Process (...,MACROD2;FHIT,GO_Biological_Process_2023,0.024333
1,Response To Calcium Ion (GO:0051592),FGB;NEDD4;FGG;CLU,GO_Biological_Process_2023,0.024333
2,Positive Regulation Of Peptide Secretion (GO:0...,FGB;FGG,GO_Biological_Process_2023,0.024333
3,Plasminogen Activation (GO:0031639),FGB;FGG,GO_Biological_Process_2023,0.024333


In [32]:
# Plot histograms of cell type distributions along top 3 eigencomponents (modules 1–3)
class_key = "perturbation"

# Prepare full dataset (train + val) if not already loaded
if 'full_data' not in globals() or 'full_labels' not in globals():
    full_loader = torch.utils.data.DataLoader(
        torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
        batch_size=len(train_dataset) + len(val_dataset),
        shuffle=False,
    )
    with torch.no_grad():
        full_batch = next(iter(full_loader))
        full_data, full_labels = full_batch

# Project all cells onto the first 3 eigencomponents (modules)
with torch.no_grad():
    top_k = 3
    component_scores = torch.matmul(full_data, vecs[:, :top_k])  # (n_cells, 3)

# Convert to numpy / pandas for plotting
scores_np = component_scores.numpy()
labels_np = full_labels.numpy()
class_names = adata.obs[class_key].cat.categories
perturbation_labels = [class_names[i] for i in labels_np.argmax(axis=1)]

# Build a long-form DataFrame for Plotly
score_rows = []
for m in range(top_k):
    for score, ctype in zip(scores_np[:, m], perturbation_labels):
        score_rows.append({
            'Module': f'Module {m+1}',
            'Score': score,
            'Perturbation': ctype,
        })
plot_df = pd.DataFrame(score_rows)

# Establish consistent (shared) bin edges across ALL histograms (modules & cell types)
# Use symmetric range around zero for fair comparison of signed projections.
extent = max(abs(scores_np.min()), abs(scores_np.max()))
# Add a small epsilon to avoid zero-width issues when extremely narrow
epsilon = 1e-9
global_min, global_max = -extent - epsilon, extent + epsilon
n_bins = 100  # adjust if you want coarser/finer resolution
bin_size = (global_max - global_min) / n_bins

# Create stacked subplots (one histogram per module)
fig = make_subplots(rows=3, cols=1, shared_xaxes=False, shared_yaxes=False,
                    vertical_spacing=0.07, subplot_titles=[f'Module {i+1}' for i in range(top_k)])

unique_perturbations = sorted(plot_df['Perturbation'].unique())

opacity = 0.75
for m in range(top_k):
    module_name = f'Module {m+1}'
    module_df = plot_df[plot_df['Module'] == module_name]
    row = m + 1
    for idx, ctype in enumerate(unique_perturbations):
        ctype_scores = module_df[module_df['Perturbation'] == ctype]['Score']
        # Use predefined cluster color; fall back to palette if missing
        marker_color = cluster_colors.get(ctype, px.colors.qualitative.Plotly[idx % len(px.colors.qualitative.Plotly)])
        fig.add_trace(
            go.Histogram(
                x=ctype_scores,
                name=ctype if m == 0 else ctype,
                legendgroup=ctype,
                opacity=opacity,
                # consistent bin edges for ALL traces
                xbins=dict(start=global_min, end=global_max, size=bin_size),
                marker_line=dict(width=0),
                marker_color=marker_color,
                showlegend=(m == 0),
                histnorm='probability',  # per-cell-type normalized
                hovertemplate=(
                    f"{module_name}<br>Perturbation: {ctype}<br>Bin: %{{x:.3f}}<br>Normalized Count: %{{y:.3f}}<extra></extra>"
                ),
            ),
            row=row, col=1
        )

# Layout adjustments
fig.update_layout(
    barmode='overlay',
    title=(
        f"Perturbation Distributions Along Top 3 {pert} Modules"\
        + "<br><span style='font-size:0.85em;'>Shared symmetric bin edges across all modules & perturbations</span>"
    ),
    height=250 * top_k + 170,
    width=fig_width,
    margin=dict(l=120, r=80, t=110, b=60),
    legend=dict(title=None, orientation='v', yanchor='top', y=0.98, xanchor='left', x=1.02),
)

# Axis titles
for m in range(top_k):
    row = m + 1
    fig.update_yaxes(title_text='Count (normalized)', row=row, col=1)
    if row == top_k:
        fig.update_xaxes(title_text='Module Value', row=row, col=1)

# Apply font size customization
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize), title=None),
)

# Tune axis title & tick font
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save outputs
fig.write_image(f"figures/perturbation/{pert}_modules_by_class.png", width=fig_width, height=800, scale=2)

In [33]:
# Perturbation strength info
pert_strength = adata.obsm["perturbation"][:,pert_idx]

# Module value info
module0_value = (adata.X @ vecs[:,0]).squeeze()
module1_value = (adata.X @ vecs[:,1]).squeeze()
module2_value = (adata.X @ vecs[:,2]).squeeze()

# Combine into DataFrame for analysis
df = pd.DataFrame({
    f"{pert}_pert_strength": pert_strength,
    f"{pert}_module0_value": module0_value,
    f"{pert}_module1_value": module1_value,
    f"{pert}_module2_value": module2_value,
    "pert_type": adata.obs["perturbation"].values,
})

pert_df = df[df["pert_type"] == pert]
del pert_df["pert_type"]
pert_df_by_strength_mean = pert_df.groupby([f"{pert}_pert_strength"]).mean()
pert_df_by_strength_std = pert_df.groupby([f"{pert}_pert_strength"]).std()

# Prepare base color derived from cluster_colors
base_color = cluster_colors.get(pert, '#1f77b4')  # fallback if missing

def to_rgba(color: str, alpha: float) -> str:
    try:
        if color.startswith('#') and len(color) == 7:
            r = int(color[1:3], 16)
            g = int(color[3:5], 16)
            b = int(color[5:7], 16)
            return f'rgba({r},{g},{b},{alpha})'
        if color.startswith('rgba('):
            inside = color[5:-1]
            parts = [p.strip() for p in inside.split(',')]
            r, g, b = parts[:3]
            return f'rgba({r},{g},{b},{alpha})'
        if color.startswith('rgb('):
            inside = color[4:-1]
            parts = [p.strip() for p in inside.split(',')]
            r, g, b = parts[:3]
            return f'rgba({r},{g},{b},{alpha})'
    except Exception:
        pass
    return color

fill_color = to_rgba(base_color, 0.20)
line_color = to_rgba(base_color, 1.0)

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[f'Module {i}' for i in range(3)],
                    horizontal_spacing=0.05)

# Precompute list of other perturbations
all_pert_types = sorted(df['pert_type'].unique())
other_perts = [p for p in all_pert_types if p != pert]

for module_idx in range(3):
    strengths = pert_df_by_strength_mean.index.values
    means = pert_df_by_strength_mean[f"{pert}_module{module_idx}_value"]
    stds = pert_df_by_strength_std[f"{pert}_module{module_idx}_value"].fillna(0)

    upper = means + stds
    lower = means - stds

    # Shaded band for selected perturbation
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([strengths, strengths[::-1]]),
            y=np.concatenate([upper, lower[::-1]]),
            fill='toself',
            fillcolor=fill_color,
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo='skip',
            showlegend=True if module_idx == 0 else False,
            name=f'{pert} ±1 SD'
        ),
        row=1, col=module_idx+1
    )

    # Mean line for selected perturbation
    fig.add_trace(
        go.Scatter(
            x=strengths,
            y=means,
            mode='lines+markers',
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            name=f'{pert}' if module_idx == 0 else f'{pert}',
            showlegend=True if module_idx == 0 else False,
            hovertemplate=(
                f"Strength: %{{x}}<br>Mean Module {module_idx} Value: %{{y:.4f}}" +
                "<br>+1 SD: %{{customdata[0]:.4f}}<br>-1 SD: %{{customdata[1]:.4f}}<extra></extra>"
            ),
            customdata=np.vstack([upper.values, lower.values]).T,
        ),
        row=1, col=module_idx+1
    )

    # Add single-point mean ±1 SD for other perturbation types at (near) zero strength with jitter
    if len(strengths) > 0:
        x_ref = strengths.min()
        x_range = strengths.max() - strengths.min() if strengths.max() != strengths.min() else 1.0
    else:
        x_ref = 0.0
        x_range = 1.0
    jitter_step = 0.01 * x_range  # small horizontal separation

    for idx_o, other in enumerate(other_perts):
        color_o = cluster_colors.get(other, '#888888')
        mask_o = df['pert_type'] == other
        col_name = f"{pert}_module{module_idx}_value"
        if col_name not in df.columns:
            continue
        vals_o = df.loc[mask_o, col_name]
        if vals_o.empty:
            continue
        mean_o = vals_o.mean()
        std_o = vals_o.std() if vals_o.count() > 1 else 0.0
        x_pos = x_ref + idx_o * jitter_step

        fig.add_trace(
            go.Scatter(
                x=[x_pos],
                y=[mean_o],
                mode='markers',
                marker=dict(size=7, color=color_o, symbol='circle'),
                name=other,
                showlegend=True if module_idx == 0 else False,
                hovertemplate=(
                    f"Perturbation: {other}<br>Ref Strength: %{{x}}<br>Mean Module {module_idx}: %{{y:.4f}}" +
                    ("<br>SD: %.4f" % std_o if std_o else "") + "<extra></extra>"
                ),
                error_y=dict(
                    type='data',
                    array=[std_o],
                    visible=True if std_o > 0 else False,
                    thickness=1,
                    width=4,
                    color=color_o
                )
            ),
            row=1, col=module_idx+1
        )

# Axis labels
for c in range(1, 4):
    fig.update_xaxes(title_text="Perturbation Strength", row=1, col=c)
fig.update_yaxes(title_text="Module Value", row=1, col=1)

fig.update_layout(
    title=dict(
        text=(
            "Perturbation Strength (log10) vs Module Values (Mean ±1 SD)"
        ),
        y=0.99,           # raise title higher (0=bottom, 1=top)
        yanchor='top',     # interpret y as the top of the title block
        pad=dict(b=8)      # a little extra space below title to clear legend
    ),
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    margin=dict(l=80, r=40, t=100, b=70),
    legend=dict(orientation='h', yanchor='bottom', y=1.12, xanchor='center', x=0.5, font=dict(size=legend_fontsize)),
    width=1400,
    height=450,
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))

fig.show()

# Save plot
fig.write_image(f"figures/perturbation/{pert}_modules_by_strength.png", width=fig_width, height=370, scale=2)

In [34]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/perturbation/{pert}_eigvals.html")
fig.write_image(f"figures/perturbation/{pert}_eigvals.png", width=fig_width, height=300, scale=2)

## Dex

In [35]:
pert_idx = 1
pert = idx2pert[pert_idx]
print(pert)

Dex


In [36]:
# Decompose weights and extract gene modules
vals, vecs = decompose_gene_weights(model, pert_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"{idx2pert[pert_idx]}", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== Dex ====================
==================== Module 0 ====================
Positive genes: ['COBLL1' 'ITGB4' 'MT2A' 'CIDEC' 'SDK2' 'FGD4' 'FKBP5' 'TFCP2L1' 'ANGPTL4'
 'CHST9']...
Negative genes: ['C5orf66' 'LINC00824' 'PHLDA2' 'PAPLN' 'CCDC175' 'PTPRG-AS1' 'MT-ATP6'
 'FAM196A' 'KCNT2' 'TCF4']...
==================== Module 1 ====================
Positive genes: ['C5orf66' 'SLPI' 'SERPINA3' 'MT-ATP6' 'GPX2' 'HSD11B2' 'FGG' 'S100P'
 'MT-ND3' 'C3']...
Negative genes: ['COBLL1' 'TP53I3' 'IGFL2-AS1' 'PLD5' 'AKAP12' 'EPN2' 'NEU1' 'ASTN2'
 'LYST' 'TNFSF15']...
==================== Module 2 ====================
Positive genes: ['FDXR' 'MDM2' 'CDKN1A' 'MORC3' 'FSTL4' 'DIAPH3' 'GALNT18' 'KRT18' 'DHRS2'
 'LINC01021']...
Negative genes: ['LUCAT1' 'PLCXD3' 'NCKAP5' 'TP63' 'ACLY' 'DPYD' 'PDK4' 'LMCD1-AS1' 'SSH2'
 'ITFG1']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
793,Hypoxia,CDKN1C;AKAP12;ERRFI1;MT2A;SERPINE1;ANGPTL4;TGM2,MSigDB_Hallmark_2020,0.000020
825,Nuclear Receptors Meta-Pathway WP2882,CDKN1C;FGD4;SERPINA1;ANGPTL4;DNAJC15;CYP3A5;FK...,WikiPathways_2019_Human,0.000094
794,Estrogen Response Early,KRT18;KRT8;SYBU;FKBP5;TGM2,MSigDB_Hallmark_2020,0.002237
515,Keratin Filament (GO:0045095),KRT18;KRT8;KRT7,GO_Cellular_Component_2023,0.006053
795,Apoptosis,KRT18;DPYD;HSPB1;BIRC3,MSigDB_Hallmark_2020,0.007449
796,Complement,C3;DOCK4;SERPINA1;SERPINE1,MSigDB_Hallmark_2020,0.012479
797,Androgen Response,AKAP12;KRT8;FKBP5,MSigDB_Hallmark_2020,0.012830
798,Bile Acid Metabolism,NEDD4;ACSL5;TFCP2L1,MSigDB_Hallmark_2020,0.014757
516,Intermediate Filament (GO:0005882),KRT18;KRT8;KRT7,GO_Cellular_Component_2023,0.016460
826,Complement and Coagulation Cascades WP558,C3;SERPINA1;SERPINE1,WikiPathways_2019_Human,0.016883



--- Negative genes ---
No significant results found (p < 0.05)
========== Module 1 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
477,Secretory Granule Lumen (GO:0034774),FGB;C3;SERPINA3;SLPI;FGG;S100P,GO_Cellular_Component_2023,0.007880
400,Neurotrophin Binding (GO:0043121),NTRK3;PCSK6,GO_Molecular_Function_2023,0.009825
0,Positive Regulation Of Cell-Substrate Adhesion...,FGB;FGG;AGR2;EMP2,GO_Biological_Process_2023,0.011872
883,Estrogen Response Early,MUC1;KRT8;AQP3;FKBP5,MSigDB_Hallmark_2020,0.013259
884,Estrogen Response Late,SERPINA3;AGR2;EMP2;FKBP5,MSigDB_Hallmark_2020,0.013259
647,MET Interacts With TNS Proteins R-HSA-8875513,TNS4;TNS3,Reactome_2022,0.014386
1,Endothelial Cell Migration (GO:0043542),STARD13;EMP2;S100P,GO_Biological_Process_2023,0.016986
2,Positive Regulation Of Peptide Secretion (GO:0...,FGB;FGG,GO_Biological_Process_2023,0.016986
3,Plasminogen Activation (GO:0031639),FGB;FGG,GO_Biological_Process_2023,0.016986
478,Platelet Alpha Granule Lumen (GO:0031093),FGB;SERPINA3;FGG,GO_Cellular_Component_2023,0.018064



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
913,Estrogen Response Late,SLC7A5;SLC9A3R1;IGSF1;CKB;AREG,MSigDB_Hallmark_2020,0.003496
914,Hypoxia,CDKN1C;AKAP12;SERPINE1;TGFBI,MSigDB_Hallmark_2020,0.009749
915,G2-M Checkpoint,SLC7A5;POLQ;CENPF;DMD,MSigDB_Hallmark_2020,0.009749
916,Estrogen Response Early,SLC7A5;SLC9A3R1;DLC1;AREG,MSigDB_Hallmark_2020,0.009749
917,UV Response Dn,DLC1;INSIG1;SERPINE1,MSigDB_Hallmark_2020,0.027974
566,Microvillus Membrane (GO:0031528),SLC7A5;SLC9A3R1,GO_Cellular_Component_2023,0.028635
918,TGF-beta Signaling,CDKN1C;SERPINE1,MSigDB_Hallmark_2020,0.033615
919,TNF-alpha Signaling via NF-kB,SERPINE1;AREG;BIRC3,MSigDB_Hallmark_2020,0.037996
920,mTORC1 Signaling,SLC7A5;SLC9A3R1;INSIG1,MSigDB_Hallmark_2020,0.037996
921,Epithelial Mesenchymal Transition,SERPINE1;TGFBI;AREG,MSigDB_Hallmark_2020,0.037996


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1107,Apoptosis,TOP2A;CDKN1A;KRT18;FDXR;BIRC3,MSigDB_Hallmark_2020,0.001206
1108,mTORC1 Signaling,SLC7A5;SLC9A3R1;CDKN1A;HMGCS1;FDXR,MSigDB_Hallmark_2020,0.001678
1131,Gastric Cancer Network 1 WP2361,TOP2A;CENPF;S100P,WikiPathways_2019_Human,0.004977
0,Regulation Of Cysteine-Type Endopeptidase Acti...,EPHA7;GRIN2B;BIRC3,GO_Biological_Process_2023,0.005102
1,Spindle Assembly Checkpoint Signaling (GO:0071...,CENPF;KNTC1;TEX14,GO_Biological_Process_2023,0.005102
2,Mitotic Spindle Assembly Checkpoint Signaling ...,CENPF;KNTC1;TEX14,GO_Biological_Process_2023,0.005102
3,Mitotic Spindle Checkpoint Signaling (GO:0071174),CENPF;KNTC1;TEX14,GO_Biological_Process_2023,0.005102
4,Negative Regulation Of Mitotic Metaphase/Anaph...,CENPF;KNTC1;TEX14,GO_Biological_Process_2023,0.005125
1111,Estrogen Response Late,TOP2A;SLC7A5;SLC9A3R1;DHRS2,MSigDB_Hallmark_2020,0.007487
1110,Estrogen Response Early,SLC7A5;SLC9A3R1;KRT18;DHRS2,MSigDB_Hallmark_2020,0.007487



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
609,Estrogen Receptor Pathway WP2881,PDK4;CYP1B1,WikiPathways_2019_Human,0.023002


In [37]:
# Plot histograms of cell type distributions along top 3 eigencomponents (modules 1–3)
class_key = "perturbation"

# Prepare full dataset (train + val) if not already loaded
if 'full_data' not in globals() or 'full_labels' not in globals():
    full_loader = torch.utils.data.DataLoader(
        torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
        batch_size=len(train_dataset) + len(val_dataset),
        shuffle=False,
    )
    with torch.no_grad():
        full_batch = next(iter(full_loader))
        full_data, full_labels = full_batch

# Project all cells onto the first 3 eigencomponents (modules)
with torch.no_grad():
    top_k = 3
    component_scores = torch.matmul(full_data, vecs[:, :top_k])  # (n_cells, 3)

# Convert to numpy / pandas for plotting
scores_np = component_scores.numpy()
labels_np = full_labels.numpy()
class_names = adata.obs[class_key].cat.categories
perturbation_labels = [class_names[i] for i in labels_np.argmax(axis=1)]

# Build a long-form DataFrame for Plotly
score_rows = []
for m in range(top_k):
    for score, ctype in zip(scores_np[:, m], perturbation_labels):
        score_rows.append({
            'Module': f'Module {m+1}',
            'Score': score,
            'Perturbation': ctype,
        })
plot_df = pd.DataFrame(score_rows)

# Establish consistent (shared) bin edges across ALL histograms (modules & cell types)
# Use symmetric range around zero for fair comparison of signed projections.
extent = max(abs(scores_np.min()), abs(scores_np.max()))
# Add a small epsilon to avoid zero-width issues when extremely narrow
epsilon = 1e-9
global_min, global_max = -extent - epsilon, extent + epsilon
n_bins = 100  # adjust if you want coarser/finer resolution
bin_size = (global_max - global_min) / n_bins

# Create stacked subplots (one histogram per module)
fig = make_subplots(rows=3, cols=1, shared_xaxes=False, shared_yaxes=False,
                    vertical_spacing=0.07, subplot_titles=[f'Module {i+1}' for i in range(top_k)])

unique_perturbations = sorted(plot_df['Perturbation'].unique())

opacity = 0.75
for m in range(top_k):
    module_name = f'Module {m+1}'
    module_df = plot_df[plot_df['Module'] == module_name]
    row = m + 1
    for idx, ctype in enumerate(unique_perturbations):
        ctype_scores = module_df[module_df['Perturbation'] == ctype]['Score']
        # Use predefined cluster color; fall back to palette if missing
        marker_color = cluster_colors.get(ctype, px.colors.qualitative.Plotly[idx % len(px.colors.qualitative.Plotly)])
        fig.add_trace(
            go.Histogram(
                x=ctype_scores,
                name=ctype if m == 0 else ctype,
                legendgroup=ctype,
                opacity=opacity,
                # consistent bin edges for ALL traces
                xbins=dict(start=global_min, end=global_max, size=bin_size),
                marker_line=dict(width=0),
                marker_color=marker_color,
                showlegend=(m == 0),
                histnorm='probability',  # per-cell-type normalized
                hovertemplate=(
                    f"{module_name}<br>Perturbation: {ctype}<br>Bin: %{{x:.3f}}<br>Normalized Count: %{{y:.3f}}<extra></extra>"
                ),
            ),
            row=row, col=1
        )

# Layout adjustments
fig.update_layout(
    barmode='overlay',
    title=(
        f"Perturbation Distributions Along Top 3 {pert} Modules"\
        + "<br><span style='font-size:0.85em;'>Shared symmetric bin edges across all modules & perturbations</span>"
    ),
    height=250 * top_k + 170,
    width=fig_width,
    margin=dict(l=120, r=80, t=110, b=60),
    legend=dict(title=None, orientation='v', yanchor='top', y=0.98, xanchor='left', x=1.02),
)

# Axis titles
for m in range(top_k):
    row = m + 1
    fig.update_yaxes(title_text='Count (normalized)', row=row, col=1)
    if row == top_k:
        fig.update_xaxes(title_text='Module Value', row=row, col=1)

# Apply font size customization
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize), title=None),
)

# Tune axis title & tick font
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save outputs
fig.write_image(f"figures/perturbation/{pert}_modules_by_class.png", width=fig_width, height=800, scale=2)

In [38]:
# Perturbation strength info
pert_strength = adata.obsm["perturbation"][:,pert_idx]

# Module value info
module0_value = (adata.X @ vecs[:,0]).squeeze()
module1_value = (adata.X @ vecs[:,1]).squeeze()
module2_value = (adata.X @ vecs[:,2]).squeeze()

# Combine into DataFrame for analysis
df = pd.DataFrame({
    f"{pert}_pert_strength": pert_strength,
    f"{pert}_module0_value": module0_value,
    f"{pert}_module1_value": module1_value,
    f"{pert}_module2_value": module2_value,
    "pert_type": adata.obs["perturbation"].values,
})

pert_df = df[df["pert_type"] == pert]
del pert_df["pert_type"]
pert_df_by_strength_mean = pert_df.groupby([f"{pert}_pert_strength"]).mean()
pert_df_by_strength_std = pert_df.groupby([f"{pert}_pert_strength"]).std()

# Prepare base color derived from cluster_colors
base_color = cluster_colors.get(pert, '#1f77b4')  # fallback if missing

def to_rgba(color: str, alpha: float) -> str:
    try:
        if color.startswith('#') and len(color) == 7:
            r = int(color[1:3], 16)
            g = int(color[3:5], 16)
            b = int(color[5:7], 16)
            return f'rgba({r},{g},{b},{alpha})'
        if color.startswith('rgba('):
            inside = color[5:-1]
            parts = [p.strip() for p in inside.split(',')]
            r, g, b = parts[:3]
            return f'rgba({r},{g},{b},{alpha})'
        if color.startswith('rgb('):
            inside = color[4:-1]
            parts = [p.strip() for p in inside.split(',')]
            r, g, b = parts[:3]
            return f'rgba({r},{g},{b},{alpha})'
    except Exception:
        pass
    return color

fill_color = to_rgba(base_color, 0.20)
line_color = to_rgba(base_color, 1.0)

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[f'Module {i}' for i in range(3)],
                    horizontal_spacing=0.05)

# Precompute list of other perturbations
all_pert_types = sorted(df['pert_type'].unique())
other_perts = [p for p in all_pert_types if p != pert]

for module_idx in range(3):
    strengths = pert_df_by_strength_mean.index.values
    means = pert_df_by_strength_mean[f"{pert}_module{module_idx}_value"]
    stds = pert_df_by_strength_std[f"{pert}_module{module_idx}_value"].fillna(0)

    upper = means + stds
    lower = means - stds

    # Shaded band for selected perturbation
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([strengths, strengths[::-1]]),
            y=np.concatenate([upper, lower[::-1]]),
            fill='toself',
            fillcolor=fill_color,
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo='skip',
            showlegend=True if module_idx == 0 else False,
            name=f'{pert} ±1 SD'
        ),
        row=1, col=module_idx+1
    )

    # Mean line for selected perturbation
    fig.add_trace(
        go.Scatter(
            x=strengths,
            y=means,
            mode='lines+markers',
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            name=f'{pert}' if module_idx == 0 else f'{pert}',
            showlegend=True if module_idx == 0 else False,
            hovertemplate=(
                f"Strength: %{{x}}<br>Mean Module {module_idx} Value: %{{y:.4f}}" +
                "<br>+1 SD: %{{customdata[0]:.4f}}<br>-1 SD: %{{customdata[1]:.4f}}<extra></extra>"
            ),
            customdata=np.vstack([upper.values, lower.values]).T,
        ),
        row=1, col=module_idx+1
    )

    # Add single-point mean ±1 SD for other perturbation types at (near) zero strength with jitter
    if len(strengths) > 0:
        x_ref = strengths.min()
        x_range = strengths.max() - strengths.min() if strengths.max() != strengths.min() else 1.0
    else:
        x_ref = 0.0
        x_range = 1.0
    jitter_step = 0.01 * x_range  # small horizontal separation

    for idx_o, other in enumerate(other_perts):
        color_o = cluster_colors.get(other, '#888888')
        mask_o = df['pert_type'] == other
        col_name = f"{pert}_module{module_idx}_value"
        if col_name not in df.columns:
            continue
        vals_o = df.loc[mask_o, col_name]
        if vals_o.empty:
            continue
        mean_o = vals_o.mean()
        std_o = vals_o.std() if vals_o.count() > 1 else 0.0
        x_pos = x_ref + idx_o * jitter_step

        fig.add_trace(
            go.Scatter(
                x=[x_pos],
                y=[mean_o],
                mode='markers',
                marker=dict(size=7, color=color_o, symbol='circle'),
                name=other,
                showlegend=True if module_idx == 0 else False,
                hovertemplate=(
                    f"Perturbation: {other}<br>Ref Strength: %{{x}}<br>Mean Module {module_idx}: %{{y:.4f}}" +
                    ("<br>SD: %.4f" % std_o if std_o else "") + "<extra></extra>"
                ),
                error_y=dict(
                    type='data',
                    array=[std_o],
                    visible=True if std_o > 0 else False,
                    thickness=1,
                    width=4,
                    color=color_o
                )
            ),
            row=1, col=module_idx+1
        )

# Axis labels
for c in range(1, 4):
    fig.update_xaxes(title_text="Perturbation Strength", row=1, col=c)
fig.update_yaxes(title_text="Module Value", row=1, col=1)

fig.update_layout(
    title=dict(
        text=(
            "Perturbation Strength (log10) vs Module Values (Mean ±1 SD)"
        ),
        y=0.99,           # raise title higher (0=bottom, 1=top)
        yanchor='top',     # interpret y as the top of the title block
        pad=dict(b=8)      # a little extra space below title to clear legend
    ),
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    margin=dict(l=80, r=40, t=100, b=70),
    legend=dict(orientation='h', yanchor='bottom', y=1.12, xanchor='center', x=0.5, font=dict(size=legend_fontsize)),
    width=1400,
    height=450,
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))

fig.show()

# Save plot
fig.write_image(f"figures/perturbation/{pert}_modules_by_strength.png", width=fig_width, height=370, scale=2)

In [39]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/perturbation/{pert}_eigvals.html")
fig.write_image(f"figures/perturbation/{pert}_eigvals.png", width=fig_width, height=300, scale=2)

## Nutlin

In [40]:
pert_idx = 2
pert = idx2pert[pert_idx]
print(pert)

Nutlin


In [41]:
# Decompose weights and extract gene modules
vals, vecs = decompose_gene_weights(model, pert_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"{idx2pert[pert_idx]}", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== Nutlin ====================
==================== Module 0 ====================
Positive genes: ['MDM2' 'CDKN1A' 'FDXR' 'TP53I3' 'FAM129A' 'ALDH3A1' 'PLD5' 'LINC01021'
 'TRANK1' 'AC025627.1']...
Negative genes: ['USP37' 'ENOX1' 'NR0B1' 'SCUBE3' 'DPYD' 'AC016205.1' 'BX470209.1'
 'PLCXD3' 'CKB' 'ROBO2']...
==================== Module 1 ====================
Positive genes: ['PDE4B' 'LINC00511' 'ALDH3A1' 'ST8SIA4' 'IGSF1' 'CSGALNACT1' 'FHIT'
 'CRY1' 'MT-RNR2' 'FTL']...
Negative genes: ['FKBP5' 'KRT18' 'DOCK4' 'C3' 'HGD' 'CIDEC' 'FGD4' 'BIRC3' 'MDM2' 'ERRFI1']...
==================== Module 2 ====================
Positive genes: ['MDM2' 'TP53I3' 'DLG2' 'KRT81' 'ALDH3A1' 'HSPA1A' 'SLC22A4' 'FBN2' 'P3H2'
 'IGFBP7']...
Negative genes: ['CA12' 'MCTP1' 'AC124319.2' 'P4HA3' 'AC010197.1' 'RHOB' 'SVEP1' 'DHRS2'
 'FGD4' 'AC138819.1']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
689,p53 signaling pathway,CDKN1A;TP53I3;MDM2;TNFRSF10B,KEGG_2019_Human,0.002196
966,Apoptosis-related network due to altered Notch...,CDKN1A;TNFRSF10B;NRG1,WikiPathways_2019_Human,0.014165
967,Genotoxicity pathway WP4286,CDKN1A;TP53I3;MDM2,WikiPathways_2019_Human,0.014165
968,DNA Damage Response WP707,CDKN1A;MDM2;TNFRSF10B,WikiPathways_2019_Human,0.014165
969,miRNA Regulation of DNA Damage Response WP1530,CDKN1A;MDM2;TNFRSF10B,WikiPathways_2019_Human,0.014165
970,TP53 Network WP1742,CDKN1A;MDM2,WikiPathways_2019_Human,0.015493
945,p53 Pathway,CDKN1A;MDM2;FDXR;NUPR1,MSigDB_Hallmark_2020,0.016379
946,mTORC1 Signaling,CDKN1A;GBE1;FDXR;NUPR1,MSigDB_Hallmark_2020,0.016379
971,ErbB Signaling Pathway WP673,CDKN1A;MDM2;NRG1,WikiPathways_2019_Human,0.017720
972,miRNA regulation of p53 pathway in prostate ca...,MDM2;TNFRSF10B,WikiPathways_2019_Human,0.017720



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
0,Cell Communication By Electrical Coupling Invo...,RYR2;ATP1A3;SLC8A1,GO_Biological_Process_2023,0.003025
1,Cellular Response To Caffeine (GO:0071313),RYR2;SLC8A1,GO_Biological_Process_2023,0.011442
2,Response To Caffeine (GO:0031000),RYR2;SLC8A1,GO_Biological_Process_2023,0.011442
3,Calcium Ion Transport Into Cytosol (GO:0060402),RYR2;SLC8A1,GO_Biological_Process_2023,0.016397
4,Cellular Response To Purine-Containing Compoun...,RYR2;SLC8A1,GO_Biological_Process_2023,0.016397
569,Ion Homeostasis R-HSA-5578775,RYR2;ATP1A3;SLC8A1,Reactome_2022,0.028383
5,Response To Muscle Stretch (GO:0035994),RYR2;SLC8A1,GO_Biological_Process_2023,0.034265
6,Regulation Of Cardiac Muscle Contraction By Re...,RYR2;SLC8A1,GO_Biological_Process_2023,0.038163
7,Cellular Response To Alkaloid (GO:0071312),RYR2;SLC8A1,GO_Biological_Process_2023,0.038163
8,Intracellular Sodium Ion Homeostasis (GO:0006883),ATP1A3;SLC8A1,GO_Biological_Process_2023,0.038163


========== Module 1 ==========

--- Positive genes ---


...

,Term,Genes,Gene_set,Adjusted P-value
0,Positive Regulation Of Vasculature Development...,CHRNA7;SERPINE1;HSPB1;EMP2;ANGPTL4;RHOB,GO_Biological_Process_2023,0.000129
1,Positive Regulation Of Angiogenesis (GO:0045766),CHRNA7;SERPINE1;HSPB1;EMP2;ANGPTL4;RHOB,GO_Biological_Process_2023,0.000143
663,Keratin Filament (GO:0045095),KRT81;KRT18;KRT8;KRT7,GO_Cellular_Component_2023,0.000155
2,Endothelial Tube Morphogenesis (GO:0061154),STARD13;RHOA;RHOB,GO_Biological_Process_2023,0.000233
1055,Estrogen Response Early,KRT18;DLC1;KRT8;SYBU;FKBP5;TGM2,MSigDB_Hallmark_2020,0.000327
664,Intermediate Filament (GO:0005882),KRT81;KRT18;KRT8;KRT7,GO_Cellular_Component_2023,0.000769
1056,Androgen Response,AKAP12;IDI1;KRT8;FKBP5,MSigDB_Hallmark_2020,0.001492
1057,Hypoxia,AKAP12;ERRFI1;SERPINE1;ANGPTL4;TGM2,MSigDB_Hallmark_2020,0.001492
3,Regulation Of Angiogenesis (GO:0045765),CHRNA7;SERPINE1;HSPB1;EMP2;ANGPTL4;RHOB,GO_Biological_Process_2023,0.001682
1058,Apoptosis,KRT18;HSPB1;RHOB;BIRC3,MSigDB_Hallmark_2020,0.005587


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
548,Adrenergic Receptor Binding (GO:0031690),SLC9A3R1;NEDD4;MAGI2,GO_Molecular_Function_2023,0.000596
549,PDZ Domain Binding (GO:0030165),SLC22A4;SLC9A3R1;DOCK4,GO_Molecular_Function_2023,0.022803
1073,p53 Pathway,ITGB4;MDM2;FDXR;TP63,MSigDB_Hallmark_2020,0.034317
550,Disordered Domain Specific Binding (GO:0097718),MDM2;HSPA1A,GO_Molecular_Function_2023,0.045455



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
0,Leucine Transport (GO:0015820),SLC7A5;SLC7A7,GO_Biological_Process_2023,0.038375
1,Branched-Chain Amino Acid Transport (GO:0015803),SLC7A5;SLC7A7,GO_Biological_Process_2023,0.038375
2,Negative Regulation Of Cell Motility (GO:2000146),FRMD5;MCTP1;STC1;RHOB,GO_Biological_Process_2023,0.038375


In [42]:
# Plot histograms of cell type distributions along top 3 eigencomponents (modules 1–3)
class_key = "perturbation"

# Prepare full dataset (train + val) if not already loaded
if 'full_data' not in globals() or 'full_labels' not in globals():
    full_loader = torch.utils.data.DataLoader(
        torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
        batch_size=len(train_dataset) + len(val_dataset),
        shuffle=False,
    )
    with torch.no_grad():
        full_batch = next(iter(full_loader))
        full_data, full_labels = full_batch

# Project all cells onto the first 3 eigencomponents (modules)
with torch.no_grad():
    top_k = 3
    component_scores = torch.matmul(full_data, vecs[:, :top_k])  # (n_cells, 3)

# Convert to numpy / pandas for plotting
scores_np = component_scores.numpy()
labels_np = full_labels.numpy()
class_names = adata.obs[class_key].cat.categories
perturbation_labels = [class_names[i] for i in labels_np.argmax(axis=1)]

# Build a long-form DataFrame for Plotly
score_rows = []
for m in range(top_k):
    for score, ctype in zip(scores_np[:, m], perturbation_labels):
        score_rows.append({
            'Module': f'Module {m+1}',
            'Score': score,
            'Perturbation': ctype,
        })
plot_df = pd.DataFrame(score_rows)

# Establish consistent (shared) bin edges across ALL histograms (modules & cell types)
# Use symmetric range around zero for fair comparison of signed projections.
extent = max(abs(scores_np.min()), abs(scores_np.max()))
# Add a small epsilon to avoid zero-width issues when extremely narrow
epsilon = 1e-9
global_min, global_max = -extent - epsilon, extent + epsilon
n_bins = 100  # adjust if you want coarser/finer resolution
bin_size = (global_max - global_min) / n_bins

# Create stacked subplots (one histogram per module)
fig = make_subplots(rows=3, cols=1, shared_xaxes=False, shared_yaxes=False,
                    vertical_spacing=0.07, subplot_titles=[f'Module {i+1}' for i in range(top_k)])

unique_perturbations = sorted(plot_df['Perturbation'].unique())

opacity = 0.75
for m in range(top_k):
    module_name = f'Module {m+1}'
    module_df = plot_df[plot_df['Module'] == module_name]
    row = m + 1
    for idx, ctype in enumerate(unique_perturbations):
        ctype_scores = module_df[module_df['Perturbation'] == ctype]['Score']
        # Use predefined cluster color; fall back to palette if missing
        marker_color = cluster_colors.get(ctype, px.colors.qualitative.Plotly[idx % len(px.colors.qualitative.Plotly)])
        fig.add_trace(
            go.Histogram(
                x=ctype_scores,
                name=ctype if m == 0 else ctype,
                legendgroup=ctype,
                opacity=opacity,
                # consistent bin edges for ALL traces
                xbins=dict(start=global_min, end=global_max, size=bin_size),
                marker_line=dict(width=0),
                marker_color=marker_color,
                showlegend=(m == 0),
                histnorm='probability',  # per-cell-type normalized
                hovertemplate=(
                    f"{module_name}<br>Perturbation: {ctype}<br>Bin: %{{x:.3f}}<br>Normalized Count: %{{y:.3f}}<extra></extra>"
                ),
            ),
            row=row, col=1
        )

# Layout adjustments
fig.update_layout(
    barmode='overlay',
    title=(
        f"Perturbation Distributions Along Top 3 {pert} Modules"\
        + "<br><span style='font-size:0.85em;'>Shared symmetric bin edges across all modules & perturbations</span>"
    ),
    height=250 * top_k + 170,
    width=fig_width,
    margin=dict(l=120, r=80, t=110, b=60),
    legend=dict(title=None, orientation='v', yanchor='top', y=0.98, xanchor='left', x=1.02),
)

# Axis titles
for m in range(top_k):
    row = m + 1
    fig.update_yaxes(title_text='Count (normalized)', row=row, col=1)
    if row == top_k:
        fig.update_xaxes(title_text='Module Value', row=row, col=1)

# Apply font size customization
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize), title=None),
)

# Tune axis title & tick font
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save outputs
fig.write_image(f"figures/perturbation/{pert}_modules_by_class.png", width=fig_width, height=800, scale=2)

In [43]:
# Perturbation strength info
pert_strength = adata.obsm["perturbation"][:,pert_idx]

# Module value info
module0_value = (adata.X @ vecs[:,0]).squeeze()
module1_value = (adata.X @ vecs[:,1]).squeeze()
module2_value = (adata.X @ vecs[:,2]).squeeze()

# Combine into DataFrame for analysis
df = pd.DataFrame({
    f"{pert}_pert_strength": pert_strength,
    f"{pert}_module0_value": module0_value,
    f"{pert}_module1_value": module1_value,
    f"{pert}_module2_value": module2_value,
    "pert_type": adata.obs["perturbation"].values,
})

pert_df = df[df["pert_type"] == pert]
del pert_df["pert_type"]
pert_df_by_strength_mean = pert_df.groupby([f"{pert}_pert_strength"]).mean()
pert_df_by_strength_std = pert_df.groupby([f"{pert}_pert_strength"]).std()

# Prepare base color derived from cluster_colors
base_color = cluster_colors.get(pert, '#1f77b4')  # fallback if missing

def to_rgba(color: str, alpha: float) -> str:
    try:
        if color.startswith('#') and len(color) == 7:
            r = int(color[1:3], 16)
            g = int(color[3:5], 16)
            b = int(color[5:7], 16)
            return f'rgba({r},{g},{b},{alpha})'
        if color.startswith('rgba('):
            inside = color[5:-1]
            parts = [p.strip() for p in inside.split(',')]
            r, g, b = parts[:3]
            return f'rgba({r},{g},{b},{alpha})'
        if color.startswith('rgb('):
            inside = color[4:-1]
            parts = [p.strip() for p in inside.split(',')]
            r, g, b = parts[:3]
            return f'rgba({r},{g},{b},{alpha})'
    except Exception:
        pass
    return color

fill_color = to_rgba(base_color, 0.20)
line_color = to_rgba(base_color, 1.0)

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[f'Module {i}' for i in range(3)],
                    horizontal_spacing=0.05)

# Precompute list of other perturbations
all_pert_types = sorted(df['pert_type'].unique())
other_perts = [p for p in all_pert_types if p != pert]

for module_idx in range(3):
    strengths = pert_df_by_strength_mean.index.values
    means = pert_df_by_strength_mean[f"{pert}_module{module_idx}_value"]
    stds = pert_df_by_strength_std[f"{pert}_module{module_idx}_value"].fillna(0)

    upper = means + stds
    lower = means - stds

    # Shaded band for selected perturbation
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([strengths, strengths[::-1]]),
            y=np.concatenate([upper, lower[::-1]]),
            fill='toself',
            fillcolor=fill_color,
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo='skip',
            showlegend=True if module_idx == 0 else False,
            name=f'{pert} ±1 SD'
        ),
        row=1, col=module_idx+1
    )

    # Mean line for selected perturbation
    fig.add_trace(
        go.Scatter(
            x=strengths,
            y=means,
            mode='lines+markers',
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            name=f'{pert}' if module_idx == 0 else f'{pert}',
            showlegend=True if module_idx == 0 else False,
            hovertemplate=(
                f"Strength: %{{x}}<br>Mean Module {module_idx} Value: %{{y:.4f}}" +
                "<br>+1 SD: %{{customdata[0]:.4f}}<br>-1 SD: %{{customdata[1]:.4f}}<extra></extra>"
            ),
            customdata=np.vstack([upper.values, lower.values]).T,
        ),
        row=1, col=module_idx+1
    )

    # Add single-point mean ±1 SD for other perturbation types at (near) zero strength with jitter
    if len(strengths) > 0:
        x_ref = strengths.min()
        x_range = strengths.max() - strengths.min() if strengths.max() != strengths.min() else 1.0
    else:
        x_ref = 0.0
        x_range = 1.0
    jitter_step = 0.01 * x_range  # small horizontal separation

    for idx_o, other in enumerate(other_perts):
        color_o = cluster_colors.get(other, '#888888')
        mask_o = df['pert_type'] == other
        col_name = f"{pert}_module{module_idx}_value"
        if col_name not in df.columns:
            continue
        vals_o = df.loc[mask_o, col_name]
        if vals_o.empty:
            continue
        mean_o = vals_o.mean()
        std_o = vals_o.std() if vals_o.count() > 1 else 0.0
        x_pos = x_ref + idx_o * jitter_step

        fig.add_trace(
            go.Scatter(
                x=[x_pos],
                y=[mean_o],
                mode='markers',
                marker=dict(size=7, color=color_o, symbol='circle'),
                name=other,
                showlegend=True if module_idx == 0 else False,
                hovertemplate=(
                    f"Perturbation: {other}<br>Ref Strength: %{{x}}<br>Mean Module {module_idx}: %{{y:.4f}}" +
                    ("<br>SD: %.4f" % std_o if std_o else "") + "<extra></extra>"
                ),
                error_y=dict(
                    type='data',
                    array=[std_o],
                    visible=True if std_o > 0 else False,
                    thickness=1,
                    width=4,
                    color=color_o
                )
            ),
            row=1, col=module_idx+1
        )

# Axis labels
for c in range(1, 4):
    fig.update_xaxes(title_text="Perturbation Strength", row=1, col=c)
fig.update_yaxes(title_text="Module Value", row=1, col=1)

fig.update_layout(
    title=dict(
        text=(
            "Perturbation Strength (log10) vs Module Values (Mean ±1 SD)"
        ),
        y=0.99,           # raise title higher (0=bottom, 1=top)
        yanchor='top',     # interpret y as the top of the title block
        pad=dict(b=8)      # a little extra space below title to clear legend
    ),
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    margin=dict(l=80, r=40, t=100, b=70),
    legend=dict(orientation='h', yanchor='bottom', y=1.12, xanchor='center', x=0.5, font=dict(size=legend_fontsize)),
    width=1400,
    height=450,
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))

fig.show()

# Save plot
fig.write_image(f"figures/perturbation/{pert}_modules_by_strength.png", width=fig_width, height=370, scale=2)

In [44]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/perturbation/{pert}_eigvals.html")
fig.write_image(f"figures/perturbation/{pert}_eigvals.png", width=fig_width, height=300, scale=2)

## SAHA

In [45]:
pert_idx = 3
pert = idx2pert[pert_idx]
print(pert)

SAHA


In [46]:
# Decompose weights and extract gene modules
vals, vecs = decompose_gene_weights(model, pert_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"{idx2pert[pert_idx]}", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== SAHA ====================
==================== Module 0 ====================
Positive genes: ['FTL' 'AC025627.1' 'DKK1' 'AC078923.1' 'LINC02045' 'CRYBG2'
 'RNF103-CHMP3' 'DNAH12' 'AC139493.2' 'LINC01885']...
Negative genes: ['ALDOC' 'STC1' 'SMPD3' 'AC008050.1' 'RHOB' 'CKB' 'NR4A1' 'ATP1A3'
 'PLA2G4C' 'ZNF815P']...
==================== Module 1 ====================
Positive genes: ['TFPI2' 'DKK1' 'PEG10' 'KRT19' 'GNGT1' 'SLC8A1' 'DNAH12' 'PDE5A'
 'FAM155A' 'MIR325HG']...
Negative genes: ['ALDH3A1' 'KRT7' 'TFPI' 'SERPINE1' 'MDM2' 'TP53I3' 'FKBP5' 'PCSK6' 'HGD'
 'NXF1']...
==================== Module 2 ====================
Positive genes: ['COL26A1' 'INSIG1' 'FDFT1' 'CEACAM20' 'IDI1' 'TNFRSF10B' 'AC093001.1'
 'HIST1H2BC' 'HMGCR' 'SQSTM1']...
Negative genes: ['PTGS2' 'DKK1' 'ALDOC' 'MDM2' 'FGA' 'TOP2A' 'NR4A1' 'TRPM3' 'SEMA3A'
 'PDK4']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
0,Negative Regulation Of Cardiac Muscle Cell Dif...,SOX6;DKK1,GO_Biological_Process_2023,0.013511
1,Negative Regulation Of Cardiocyte Differentiat...,SOX6;DKK1,GO_Biological_Process_2023,0.013511
2,Regulation Of Cardiac Muscle Cell Differentiat...,SOX6;DKK1,GO_Biological_Process_2023,0.021514



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
1070,Hypoxia,EXT1;AKAP12;LXN;STC1;ALDOC,MSigDB_Hallmark_2020,0.001119
1071,Estrogen Response Early,SLC9A3R1;MUC1;NAV2;AQP3;DHRS2,MSigDB_Hallmark_2020,0.001119
1072,Myogenesis,TGFB1;NAV2;SORBS1;CKB;LPIN1,MSigDB_Hallmark_2020,0.001119
1073,TNF-alpha Signaling via NF-kB,NR4A1;PHLDA2;RHOB;BIRC3,MSigDB_Hallmark_2020,0.007487
1074,Estrogen Response Late,CHST8;SLC9A3R1;CKB;DHRS2,MSigDB_Hallmark_2020,0.007487
0,Negative Regulation Of Cell Cycle (GO:0045786),SLC9A3R1;NR4A1;TGFB1;RHOB,GO_Biological_Process_2023,0.028300
1,Gland Morphogenesis (GO:0022612),SLC9A3R1;TGFB1,GO_Biological_Process_2023,0.028300
1075,UV Response Up,NR4A1;AQP3;RHOB,MSigDB_Hallmark_2020,0.028903


========== Module 1 ==========

--- Positive genes ---


...

,Term,Genes,Gene_set,Adjusted P-value
744,Hypoxia,CA12;AKAP12;ERRFI1;GBE1;SERPINE1;STC1;ANGPTL4,MSigDB_Hallmark_2020,0.000019
775,Nuclear Receptors Meta-Pathway WP2882,SLC7A5;ALDH3A1;GPX2;GCLC;ABCB1;ANGPTL4;DNAJC15...,WikiPathways_2019_Human,0.000087
745,mTORC1 Signaling,SLC7A5;GCLC;GBE1;FDXR;STC1,MSigDB_Hallmark_2020,0.002167
747,KRAS Signaling Up,AKAP12;ABCB1;ANGPTL4;TFPI,MSigDB_Hallmark_2020,0.012089
746,Estrogen Response Late,CA12;CHST8;SLC7A5;FKBP5,MSigDB_Hallmark_2020,0.012089
0,Leucine Transport (GO:0015820),SLC7A5;SLC7A7,GO_Biological_Process_2023,0.032473
2,Branched-Chain Amino Acid Transport (GO:0015803),SLC7A5;SLC7A7,GO_Biological_Process_2023,0.032473
1,Negative Regulation Of Hemostasis (GO:1900047),SERPINE1;TFPI,GO_Biological_Process_2023,0.032473
3,Negative Regulation Of Coagulation (GO:0050819),SERPINE1;TFPI,GO_Biological_Process_2023,0.035607
776,Glutathione metabolism WP100,GPX2;GCLC,WikiPathways_2019_Human,0.041347


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
673,Cholesterol Biosynthesis R-HSA-191273,IDI1;SREBF1;HMGCR;FDFT1,Reactome_2022,0.000047
672,Regulation Of Cholesterol Biosynthesis By SREB...,IDI1;SREBF1;INSIG1;HMGCR;FDFT1,Reactome_2022,0.000047
887,Sterol Regulatory Element-Binding Proteins (SR...,IDI1;SREBF1;INSIG1;HMGCR;FDFT1,WikiPathways_2019_Human,0.000050
888,Cholesterol Biosynthesis Pathway WP197,IDI1;HMGCR;FDFT1,WikiPathways_2019_Human,0.000206
674,Activation Of Gene Expression By SREBF (SREBP)...,IDI1;SREBF1;HMGCR;FDFT1,Reactome_2022,0.000230
863,Cholesterol Homeostasis,IDI1;HMGCR;CLU;FDFT1,MSigDB_Hallmark_2020,0.000839
864,Androgen Response,IDI1;INSIG1;HMGCR;FKBP5,MSigDB_Hallmark_2020,0.001363
675,Metabolism Of Steroids R-HSA-8957322,IDI1;SREBF1;INSIG1;HMGCR;FDFT1,Reactome_2022,0.001881
0,Secondary Alcohol Biosynthetic Process (GO:190...,INSIG1;HMGCR;FDFT1,GO_Biological_Process_2023,0.005107
2,Cholesterol Biosynthetic Process (GO:0006695),INSIG1;HMGCR;FDFT1,GO_Biological_Process_2023,0.005107



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
701,Intracellular Organelle Lumen (GO:0070013),FGA;ERBB4;FGG;COL4A6;PDK4;COL4A5;MUC13;ALDOC;P...,GO_Cellular_Component_2023,0.001225
703,Endoplasmic Reticulum Lumen (GO:0005788),FGA;FGG;COL4A6;COL4A5;PTGS2;F5,GO_Cellular_Component_2023,0.001225
702,Platelet Alpha Granule (GO:0031091),FGA;FGG;NRG1;F5,GO_Cellular_Component_2023,0.001225
700,Platelet Alpha Granule Lumen (GO:0031093),FGA;FGG;NRG1;F5,GO_Cellular_Component_2023,0.001225
842,Signal Transduction R-HSA-162582,ALK;FGA;RRAD;FGG;NRG1;NTS;DKK1;RHOB;SMPD3;NR4A...,Reactome_2022,0.002040
841,Diseases Of Signal Transduction By Growth Fact...,ALK;FGA;NR4A1;ERBB4;FGG;MDM2;NRG1;DKK1,Reactome_2022,0.002040
843,Common Pathway Of Fibrin Clot Formation R-HSA-...,FGA;FGG;F5,Reactome_2022,0.002043
1149,Blood Clotting Cascade WP272,FGA;FGG;F5,WikiPathways_2019_Human,0.002312
768,Small cell lung cancer,COL4A6;COL4A5;PTGS2;BIRC3,KEGG_2019_Human,0.003169
767,Complement and coagulation cascades,FGA;FGG;CD55;F5,KEGG_2019_Human,0.003169


In [47]:
# Plot histograms of cell type distributions along top 3 eigencomponents (modules 1–3)
class_key = "perturbation"

# Prepare full dataset (train + val) if not already loaded
if 'full_data' not in globals() or 'full_labels' not in globals():
    full_loader = torch.utils.data.DataLoader(
        torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
        batch_size=len(train_dataset) + len(val_dataset),
        shuffle=False,
    )
    with torch.no_grad():
        full_batch = next(iter(full_loader))
        full_data, full_labels = full_batch

# Project all cells onto the first 3 eigencomponents (modules)
with torch.no_grad():
    top_k = 3
    component_scores = torch.matmul(full_data, vecs[:, :top_k])  # (n_cells, 3)

# Convert to numpy / pandas for plotting
scores_np = component_scores.numpy()
labels_np = full_labels.numpy()
class_names = adata.obs[class_key].cat.categories
perturbation_labels = [class_names[i] for i in labels_np.argmax(axis=1)]

# Build a long-form DataFrame for Plotly
score_rows = []
for m in range(top_k):
    for score, ctype in zip(scores_np[:, m], perturbation_labels):
        score_rows.append({
            'Module': f'Module {m+1}',
            'Score': score,
            'Perturbation': ctype,
        })
plot_df = pd.DataFrame(score_rows)

# Establish consistent (shared) bin edges across ALL histograms (modules & cell types)
# Use symmetric range around zero for fair comparison of signed projections.
extent = max(abs(scores_np.min()), abs(scores_np.max()))
# Add a small epsilon to avoid zero-width issues when extremely narrow
epsilon = 1e-9
global_min, global_max = -extent - epsilon, extent + epsilon
n_bins = 100  # adjust if you want coarser/finer resolution
bin_size = (global_max - global_min) / n_bins

# Create stacked subplots (one histogram per module)
fig = make_subplots(rows=3, cols=1, shared_xaxes=False, shared_yaxes=False,
                    vertical_spacing=0.07, subplot_titles=[f'Module {i+1}' for i in range(top_k)])

unique_perturbations = sorted(plot_df['Perturbation'].unique())

opacity = 0.75
for m in range(top_k):
    module_name = f'Module {m+1}'
    module_df = plot_df[plot_df['Module'] == module_name]
    row = m + 1
    for idx, ctype in enumerate(unique_perturbations):
        ctype_scores = module_df[module_df['Perturbation'] == ctype]['Score']
        # Use predefined cluster color; fall back to palette if missing
        marker_color = cluster_colors.get(ctype, px.colors.qualitative.Plotly[idx % len(px.colors.qualitative.Plotly)])
        fig.add_trace(
            go.Histogram(
                x=ctype_scores,
                name=ctype if m == 0 else ctype,
                legendgroup=ctype,
                opacity=opacity,
                # consistent bin edges for ALL traces
                xbins=dict(start=global_min, end=global_max, size=bin_size),
                marker_line=dict(width=0),
                marker_color=marker_color,
                showlegend=(m == 0),
                histnorm='probability',  # per-cell-type normalized
                hovertemplate=(
                    f"{module_name}<br>Perturbation: {ctype}<br>Bin: %{{x:.3f}}<br>Normalized Count: %{{y:.3f}}<extra></extra>"
                ),
            ),
            row=row, col=1
        )

# Layout adjustments
fig.update_layout(
    barmode='overlay',
    title=(
        f"Perturbation Distributions Along Top 3 {pert} Modules"\
        + "<br><span style='font-size:0.85em;'>Shared symmetric bin edges across all modules & perturbations</span>"
    ),
    height=250 * top_k + 170,
    width=fig_width,
    margin=dict(l=120, r=80, t=110, b=60),
    legend=dict(title=None, orientation='v', yanchor='top', y=0.98, xanchor='left', x=1.02),
)

# Axis titles
for m in range(top_k):
    row = m + 1
    fig.update_yaxes(title_text='Count (normalized)', row=row, col=1)
    if row == top_k:
        fig.update_xaxes(title_text='Module Value', row=row, col=1)

# Apply font size customization
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize), title=None),
)

# Tune axis title & tick font
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save outputs
fig.write_image(f"figures/perturbation/{pert}_modules_by_class.png", width=fig_width, height=800, scale=2)

In [48]:
# Perturbation strength info
pert_strength = adata.obsm["perturbation"][:,pert_idx]

# Module value info
module0_value = (adata.X @ vecs[:,0]).squeeze()
module1_value = (adata.X @ vecs[:,1]).squeeze()
module2_value = (adata.X @ vecs[:,2]).squeeze()

# Combine into DataFrame for analysis
df = pd.DataFrame({
    f"{pert}_pert_strength": pert_strength,
    f"{pert}_module0_value": module0_value,
    f"{pert}_module1_value": module1_value,
    f"{pert}_module2_value": module2_value,
    "pert_type": adata.obs["perturbation"].values,
})

pert_df = df[df["pert_type"] == pert]
del pert_df["pert_type"]
pert_df_by_strength_mean = pert_df.groupby([f"{pert}_pert_strength"]).mean()
pert_df_by_strength_std = pert_df.groupby([f"{pert}_pert_strength"]).std()

# Prepare base color derived from cluster_colors
base_color = cluster_colors.get(pert, '#1f77b4')  # fallback if missing

def to_rgba(color: str, alpha: float) -> str:
    try:
        if color.startswith('#') and len(color) == 7:
            r = int(color[1:3], 16)
            g = int(color[3:5], 16)
            b = int(color[5:7], 16)
            return f'rgba({r},{g},{b},{alpha})'
        if color.startswith('rgba('):
            inside = color[5:-1]
            parts = [p.strip() for p in inside.split(',')]
            r, g, b = parts[:3]
            return f'rgba({r},{g},{b},{alpha})'
        if color.startswith('rgb('):
            inside = color[4:-1]
            parts = [p.strip() for p in inside.split(',')]
            r, g, b = parts[:3]
            return f'rgba({r},{g},{b},{alpha})'
    except Exception:
        pass
    return color

fill_color = to_rgba(base_color, 0.20)
line_color = to_rgba(base_color, 1.0)

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[f'Module {i}' for i in range(3)],
                    horizontal_spacing=0.05)

# Precompute list of other perturbations
all_pert_types = sorted(df['pert_type'].unique())
other_perts = [p for p in all_pert_types if p != pert]

for module_idx in range(3):
    strengths = pert_df_by_strength_mean.index.values
    means = pert_df_by_strength_mean[f"{pert}_module{module_idx}_value"]
    stds = pert_df_by_strength_std[f"{pert}_module{module_idx}_value"].fillna(0)

    upper = means + stds
    lower = means - stds

    # Shaded band for selected perturbation
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([strengths, strengths[::-1]]),
            y=np.concatenate([upper, lower[::-1]]),
            fill='toself',
            fillcolor=fill_color,
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo='skip',
            showlegend=True if module_idx == 0 else False,
            name=f'{pert} ±1 SD'
        ),
        row=1, col=module_idx+1
    )

    # Mean line for selected perturbation
    fig.add_trace(
        go.Scatter(
            x=strengths,
            y=means,
            mode='lines+markers',
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            name=f'{pert}' if module_idx == 0 else f'{pert}',
            showlegend=True if module_idx == 0 else False,
            hovertemplate=(
                f"Strength: %{{x}}<br>Mean Module {module_idx} Value: %{{y:.4f}}" +
                "<br>+1 SD: %{{customdata[0]:.4f}}<br>-1 SD: %{{customdata[1]:.4f}}<extra></extra>"
            ),
            customdata=np.vstack([upper.values, lower.values]).T,
        ),
        row=1, col=module_idx+1
    )

    # Add single-point mean ±1 SD for other perturbation types at (near) zero strength with jitter
    if len(strengths) > 0:
        x_ref = strengths.min()
        x_range = strengths.max() - strengths.min() if strengths.max() != strengths.min() else 1.0
    else:
        x_ref = 0.0
        x_range = 1.0
    jitter_step = 0.01 * x_range  # small horizontal separation

    for idx_o, other in enumerate(other_perts):
        color_o = cluster_colors.get(other, '#888888')
        mask_o = df['pert_type'] == other
        col_name = f"{pert}_module{module_idx}_value"
        if col_name not in df.columns:
            continue
        vals_o = df.loc[mask_o, col_name]
        if vals_o.empty:
            continue
        mean_o = vals_o.mean()
        std_o = vals_o.std() if vals_o.count() > 1 else 0.0
        x_pos = x_ref + idx_o * jitter_step

        fig.add_trace(
            go.Scatter(
                x=[x_pos],
                y=[mean_o],
                mode='markers',
                marker=dict(size=7, color=color_o, symbol='circle'),
                name=other,
                showlegend=True if module_idx == 0 else False,
                hovertemplate=(
                    f"Perturbation: {other}<br>Ref Strength: %{{x}}<br>Mean Module {module_idx}: %{{y:.4f}}" +
                    ("<br>SD: %.4f" % std_o if std_o else "") + "<extra></extra>"
                ),
                error_y=dict(
                    type='data',
                    array=[std_o],
                    visible=True if std_o > 0 else False,
                    thickness=1,
                    width=4,
                    color=color_o
                )
            ),
            row=1, col=module_idx+1
        )

# Axis labels
for c in range(1, 4):
    fig.update_xaxes(title_text="Perturbation Strength", row=1, col=c)
fig.update_yaxes(title_text="Module Value", row=1, col=1)

fig.update_layout(
    title=dict(
        text=(
            "Perturbation Strength (log10) vs Module Values (Mean ±1 SD)"
        ),
        y=0.99,           # raise title higher (0=bottom, 1=top)
        yanchor='top',     # interpret y as the top of the title block
        pad=dict(b=8)      # a little extra space below title to clear legend
    ),
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    margin=dict(l=80, r=40, t=100, b=70),
    legend=dict(orientation='h', yanchor='bottom', y=1.12, xanchor='center', x=0.5, font=dict(size=legend_fontsize)),
    width=1400,
    height=450,
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))

fig.show()

# Save plot
fig.write_image(f"figures/perturbation/{pert}_modules_by_strength.png", width=fig_width, height=370, scale=2)

In [49]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/perturbation/{pert}_eigvals.html")
fig.write_image(f"figures/perturbation/{pert}_eigvals.png", width=fig_width, height=300, scale=2)